# 01 · Entendimento do Negócio e Definição da Target

**Case Preditivo de NPS: Tech Challenge Fase 1 (Pós Tech IA para Cientistas de Dados)**

Este notebook cobre a primeira fase do **CRISP-DM**, o *Business Understanding*, e corresponde às duas primeiras etapas do desafio:

1. **Entendimento do negócio**: qual dor estamos resolvendo, por que o NPS importa e quem se beneficia;
2. **Definição da target**: qual variável representa a satisfação, quando ela é coletada e quais os riscos de usá-la mal.

Como recomendam as apostilas, **a ciência de dados começa na pergunta, não no algoritmo**. Por isso esta etapa é conceitual: o código aparece apenas para ilustrar a discussão.

## 1. Entendimento do negócio

### 1.1 A dor de negócio

A empresa cresceu rápido no e-commerce e, com mais pedidos, entregas e contatos, a experiência do cliente ficou **desigual**: clientes com indicadores operacionais parecidos terminam a jornada como promotores ou como detratores.

Hoje o NPS só é medido **depois** do fim da jornada. Quando a nota chega, o cliente já teve o atraso, já ligou várias vezes para o SAC, já reclamou. O objetivo é **transformar o NPS de um indicador de retrovisor em um indicador de para-brisa**: usar os dados operacionais que a empresa já coleta durante a jornada para antecipar a insatisfação e priorizar ações corretivas.

### 1.2 Que tipo de problema de dados é este?

Seguindo a classificação vista em aula (descritivo, diagnóstico, preditivo e prescritivo), a dor atravessa três níveis, em etapas naturais:

| Nível | Pergunta | Onde é respondida |
|---|---|---|
| **Descritivo / diagnóstico** | O que acontece com a satisfação e quais fatores operacionais estão associados a ela? | Análise exploratória (notebooks 02 e 03) |
| **Preditivo** | Qual nota este cliente tende a dar, antes da pesquisa? | Modelo de regressão (notebook 04) |
| **Prescritivo** | O que fazer com cada pedido em risco? | Faixas de risco e recomendações (notebook 04 e apresentação) |

### 1.3 Por que o NPS é importante para um e-commerce?

- **É um termômetro simples e comparável.** Uma única pergunta ("de 0 a 10, quanto você recomendaria?") resume a percepção do cliente sobre toda a jornada e permite comparar períodos, regiões e concorrentes.
- **No e-commerce, o cliente não "vê" a loja: ele vê a entrega e o atendimento.** A experiência física se resume a: o produto chegou no prazo? Quando tive problema, fui bem atendido? O NPS captura exatamente essa percepção.
- **Trocar de loja custa um clique.** A barreira de troca é baixíssima; a lealdade precisa ser conquistada a cada pedido.
- **Está ligado ao crescimento.** Promotores compram de novo e indicam; detratores deixam de comprar e falam mal, muitas vezes em público.

### 1.4 Quais áreas se beneficiam desses insights?

| Área | Como usa os insights |
|---|---|
| **Logística** | Identificar as rotas e transportadoras que mais atrasam para corrigi-las (cobrar SLA, renegociar ou substituir) e dar preferência às mais pontuais; definir prazos realistas; agir quando o pedido entra em risco de atraso. |
| **Atendimento (SAC)** | Atuar proativamente com clientes em risco; reduzir recontatos; priorizar a fila por risco de detração. |
| **Produto / Marketplace** | Identificar produtos e sellers que geram mais reclamações. |
| **Pricing / Comercial** | Calibrar frete, descontos e compensações, e saber quando *não* adianta dar desconto. |
| **CRM / Marketing** | Recuperar clientes em risco; convidar prováveis promotores para programas de indicação. |
| **Estratégia / Diretoria** | Acompanhar um "NPS previsto" e priorizar investimentos onde o retorno em satisfação é maior. |

### 1.5 Reflexão: como o NPS impacta o negócio

**Recompra.** O cliente satisfeito volta. No e-commerce, conquistar um cliente novo (mídia paga, cupom de primeira compra) costuma custar mais do que manter um cliente atual. Na própria base deste case a relação é visível: **a recompra em 30 dias muda completamente entre detratores, neutros e promotores** (ver célula abaixo).

**Boca a boca.** O NPS mede justamente a *intenção de recomendar*. Promotores viram canal de aquisição gratuito (indicações, avaliações positivas). Detratores têm o efeito contrário, geralmente mais forte: uma experiência ruim é contada com mais frequência e fica registrada em público (avaliações, sites de reclamação, redes sociais), afetando quem ainda nem comprou.

**Market share.** Com baixa barreira de troca, a participação de mercado depende de três efeitos que o NPS influencia: (1) **retenção**: quem fica compra mais; (2) **aquisição orgânica**: indicações reduzem o custo de aquisição; (3) **reputação**: notas públicas influenciam a escolha entre lojas que vendem o mesmo produto.

### 1.6 Indicadores de mercado que poderiam complementar a análise

- **Benchmarks de NPS do setor** (varejo online e marketplaces), para saber se o NPS é bom ou ruim *em relação ao mercado*;
- **Reputação pública**: nota e índice de solução em sites de reclamação, avaliações em marketplaces e lojas de aplicativo;
- **SLA logístico de mercado**: prazo médio dos concorrentes por região, % de entregas no prazo das transportadoras;
- **Concorrência**: preço e frete dos concorrentes, políticas de troca e devolução;
- **Indicadores internos**: taxa de recompra, churn, CAC, LTV, taxa de devolução, volume de reclamações por mil pedidos.

In [1]:
# Configuração: permite importar o módulo src/ a partir da pasta notebooks/
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.funcoes_nps import calcular_nps, carregar_dados_brutos, classificar_nps

dados = carregar_dados_brutos()
dados["categoria_nps"] = classificar_nps(dados["nps_score"])

# Ilustração do item 1.5: recompra em 30 dias por categoria de NPS
recompra_por_categoria = dados.groupby("categoria_nps", observed=True)["repeat_purchase_30d"].mean() * 100
recompra_por_categoria.round(1).rename("% com recompra em 30 dias").to_frame()

,% com recompra em 30 dias
categoria_nps,
Detrator,0.0
Neutro,38.4
Promotor,100.0


**O que a tabela mostra:** nenhum detrator recomprou em 30 dias, enquanto todos os promotores recompraram. A ligação entre satisfação e receita futura é direta. Mas o mesmo resultado acende um alerta: uma variável tão "colada" ao NPS, e medida *depois* dele, não pode ser usada para prevê-lo (ver item 2.5).

### 1.7 Metas de negócio × metas analíticas

As apostilas reforçam que **métricas técnicas são o caminho e os KPIs de negócio são o destino**. Definimos as duas desde o início:

| Tipo | Meta | Como medir |
|---|---|---|
| **Meta de negócio** | Reduzir a proporção de detratores e elevar o NPS, agindo antes da pesquisa | % de detratores e NPS mensal, comparando clientes tratados com um grupo de controle (teste A/B) |
| **Meta de negócio (insight)** | Entender quais fatores operacionais movem a satisfação | Fatores identificados e validados com as áreas de Logística e Atendimento |
| **Meta analítica** | Prever a nota de cada pedido com erro bem menor que um "chute" pela média | MAE (erro médio, em pontos de nota) no conjunto de teste, comparado ao baseline |
| **Meta analítica** | Separar bem quem vai virar detrator | *Recall* e precisão de detratores e AUC; % real de detratores em cada faixa de risco |

**Critério de sucesso combinado:** o modelo só é útil se, além das métricas técnicas, as faixas de risco gerarem uma fila de ação que a operação consiga executar.

### 1.8 Hipóteses iniciais (antes de olhar os dados)

Como sugere o fluxo "dor → perguntas → hipóteses → dados", registramos o que esperamos encontrar. A EDA (notebook 03) vai confirmar ou refutar cada uma:

| # | Hipótese |
|---|---|
| H1 | Atrasos na entrega reduzem a nota. |
| H2 | Quanto mais contatos com o SAC e reclamações, menor a nota. |
| H3 | Entregas mais demoradas (tempo total) geram notas menores. |
| H4 | Frete caro e pedidos de valor alto deixam o cliente mais exigente (nota menor). |
| H5 | O perfil do cliente (região, idade, tempo de casa) influencia a nota. |

## 2. Definição da target

### 2.1 Qual variável representa a satisfação do cliente?

A variável alvo é **`nps_score`**: a nota de 0 a 10 que o cliente dá à pergunta de recomendação, coletada após a experiência de compra.

Usando a fórmula vista em aula, **"[Unidade] irá [Ação] nos próximos [Período]"**, a pergunta do modelo fica:

> **Qual nota o cliente deste *pedido* (unidade) dará na *pesquisa de NPS* (ação) aplicada *ao final da jornada de compra* (período)?**

A partir da nota derivamos duas visões:

| Visão | Definição | Uso |
|---|---|---|
| **Nota contínua** | `nps_score` (0 a 10) | Modelo de regressão, ranking de risco, médias por grupo |
| **Categoria NPS** | Detrator (< 7) · Neutro (7 a < 9) · Promotor (≥ 9) | Cálculo do índice NPS e comunicação com o negócio |

> **Observação sobre as notas decimais.** Na metodologia clássica a nota é inteira (0-6 detrator, 7-8 neutro, 9-10 promotor). Nesta base a nota tem casas decimais (ex.: 6,9). Optamos por limites contínuos (**abaixo de 7 é detrator**) para não "promover" clientes como 6,9 a neutros.

### 2.2 Por que ela foi escolhida?

- É **a métrica que a empresa já acompanha** e na qual a área de Experiência do Cliente é cobrada;
- Mede **lealdade e intenção de recomendar**, mais ligadas a crescimento do que uma satisfação pontual;
- É a **visão do cliente**, não da operação: um pedido pode estar "dentro do SLA" e ainda assim gerar um detrator. Por isso é a melhor referência para descobrir quais fatores operacionais importam;
- Alternativas descartadas como alvo principal: **`csat_internal_score`** é um score interno (não sabemos como nem quando é calculado) e **`repeat_purchase_30d`** é um *resultado* que só se conhece 30 dias depois.

### 2.3 Em que momento da jornada essa informação é coletada?

Ao **final da jornada**, depois da entrega e de eventuais contatos com o atendimento:

```
Compra → Pagamento → Transporte → Entrega (atraso? tentativas?) → Atendimento / Reclamação → [Pesquisa NPS] → Recompra (30 dias)
```

É justamente esse posicionamento que motiva o case: todas as informações que aparecem *antes* da pesquisa podem servir de sinal de alerta.

### 2.4 A nota é um bom "proxy" de satisfação?

As apostilas citam o próprio NPS como exemplo de **proxy**: um substituto conveniente para algo difícil de medir (a satisfação real). Ele é útil, mas imperfeito: mede a *intenção* de recomendar, em um único momento, e só de quem respondeu. Por isso deve ser reavaliado com outros sinais (recompra, reclamações, avaliações públicas).

### 2.5 Existe risco de usar essa variável de forma inadequada?

Sim. Os principais:

1. **Vazamento de informação (*data leakage*).** `repeat_purchase_30d` (medida 30 dias depois) e `csat_internal_score` (outra medida de satisfação, provavelmente coletada junto) são muito ligadas ao NPS e "inflariam" um modelo. Ele pareceria ótimo no teste e seria inútil na prática, porque essas informações não existem no momento da previsão. **Decisão: elas ficam fora do modelo** e servem apenas para validação de negócio.
2. **Viés de seleção / autosseleção.** Só quem respondeu à pesquisa está na base, e quem responde costuma estar nos extremos (muito satisfeito ou muito irritado).
3. **Confundir correlação com causa.** Ver que quem fala mais com o SAC dá nota menor não significa que "atender menos" melhora o NPS: o contato é sintoma de um problema anterior.
4. **Transformar o NPS em meta individual.** Se o NPS virar meta de bônus, surgem incentivos para "pedir nota" ou filtrar quem recebe a pesquisa, e a métrica deixa de medir a realidade.
5. **Olhar só o índice agregado.** O NPS (% promotores − % detratores) esconde a distribuição: por isso analisamos também a nota média e a % de detratores.
6. **Target drift e definições que mudam.** Se a empresa mudar a forma de aplicar a pesquisa (canal, momento, escala), o modelo treinado com a definição antiga perde validade.
7. **Usar a previsão para punir ou discriminar clientes.** O modelo deve servir para *cuidar* de quem está em risco, nunca para tratar pior quem "provavelmente vai reclamar".

In [2]:
# Visão geral da target: distribuição das categorias e NPS da base
participacao_categorias = dados["categoria_nps"].value_counts(normalize=True).sort_index() * 100
print("Distribuição das categorias (%):")
print(participacao_categorias.round(1).to_string())
print(f"\nNota média: {dados['nps_score'].mean():.2f}")
print(f"NPS da base: {calcular_nps(dados['nps_score']):.1f} pontos")

Distribuição das categorias (%):
categoria_nps
Detrator    84.4
Neutro      11.2
Promotor     4.4

Nota média: 4.38
NPS da base: -80.0 pontos


**Leitura rápida:** a base retrata uma operação em situação crítica: a grande maioria dos pedidos termina com detratores e o NPS é fortemente negativo. O desafio não é "otimizar" um NPS bom, é **estancar a geração de detratores**. Os próximos notebooks investigam de onde eles vêm.